# Part 1 — Dataset Discovery and Ranking

## Imports and Initialization

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "part1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://data.gov.au/data/api/3/action/package_search"

print("Project:", PROJECT_ROOT)
print("Output:", OUTPUT_DIR)

Project: /Users/ario/Documents/firmable
Output: /Users/ario/Documents/firmable/outputs/part1


## Retrieve Datasets

In [2]:
QUERY = "ABN OR ACN OR business"

params = {
    "q": QUERY,
    "rows": 1000,
}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=60,
)

response.raise_for_status()

data = response.json()

print("Success:", data["success"])
print("Total matches:", data["result"]["count"])

datasets = data["result"]["results"]

print("Retrieved:", len(datasets))

Success: True
Total matches: 5689
Retrieved: 1000


## Filter Useful Resource Formats

In [3]:
USEFUL_FORMATS = {
    "CSV",
    "TSV",
    "XLS",
    "XLSX",
    "JSON",
    "GEOJSON",
    "XML",
    "ZIP",
}


def normalize_format(fmt):
    fmt = (fmt or "").strip().upper()

    aliases = {
        ".CSV": "CSV",
        ".XLS": "XLS",
        ".XLSX": "XLSX",
        "EXCEL (.XLSX)": "XLSX",
        "EXCEL (XLSX)": "XLSX",
        "MICROSOFT EXCEL": "XLSX",
        "XSLX": "XLSX",
        "CSV-GEO-AU": "CSV",
    }

    return aliases.get(fmt, fmt)


def has_useful_resource(dataset):
    return any(
        normalize_format(resource.get("format")) in USEFUL_FORMATS
        for resource in dataset.get("resources", [])
    )


datasets = [
    dataset
    for dataset in datasets
    if has_useful_resource(dataset)
]

print("Datasets with useful resources:", len(datasets))

Datasets with useful resources: 457


## Generate Dataset DataFrame

In [4]:
records = []

for ds in datasets:

    organization = ds.get("organization") or {}
    resources = ds.get("resources") or []

    tags = [
        tag.get("name", "")
        for tag in ds.get("tags", [])
    ]

    useful_resources = [
        r for r in resources
        if normalize_format(r.get("format")) in USEFUL_FORMATS
    ]

    resource_formats = sorted({
        normalize_format(r.get("format"))
        for r in useful_resources
    })

    resource_names = [
        r.get("name", "")
        for r in useful_resources
        if r.get("name")
    ]

    resource_descriptions = [
        r.get("description", "")
        for r in useful_resources
        if r.get("description")
    ]

    records.append({
        "dataset_id": ds.get("id"),
        "title": ds.get("title") or "",
        "description": ds.get("notes") or "",
        "publisher": organization.get("title") or "",
        "tags": " ".join(tags),
        "resource_names": " ".join(resource_names),
        "resource_descriptions": " ".join(resource_descriptions),
        "formats": ", ".join(resource_formats),
        "metadata_modified": ds.get("metadata_modified"),
    })


datasets_df = pd.DataFrame(records)

datasets_df.head()

,dataset_id,title,description,publisher,tags,resource_names,resource_descriptions,formats,metadata_modified
0,5bd7fcab-e315-42cb-8daf-50b7efc2027e,ABN Bulk Extract,ABN Lookup Bulk Data - Australian Business Reg...,Australian Business Register,ABN Lookup Bulk extract ABR Public Data Austra...,Bulk Extract Schema ABN Bulk Extract Part 1 AB...,Schema (xsd) for the ABN Lookup Bulk Extract f...,"CSV, XML, ZIP",2026-09-29T22:37:09.389618
1,3a3661a8-ee5d-4165-be5c-018634ca7202,ABN Lookup web services,The ABN Lookup web services allow you to integ...,Australian Business Register,ABN DGR GST SOAP web services,ABN Lookup web services,Use the ABN Lookup web services to search by t...,XML,2023-08-11T01:32:21.646547
2,1ca29b03-220d-4c19-b34e-c7932cd4508d,Victorian Government Schools ABNs,The Australian Business Number (ABN) is a uniq...,Department of Education,ABN Education School Number Schools Victorian ...,Victorian Government Schools ABNs,Further details about this specific data resou...,XLSX,2025-06-25T17:38:22.490113
3,7c356598-b048-4c1e-862f-c936041e3d81,Research and Development tax incentive,Data about Research and Development tax incent...,Australian Taxation Office,expenditure research research and development ...,2023-24 Report of data about Research and Deve...,This report contains the total research and de...,XLSX,2026-09-22T23:31:37.257725
4,42f46baa-b642-48b2-9f26-fc24d09d9320,Industry breakdown of PPSR registrations,The Personal Property Securities Register (PPS...,Australian Financial Security Authority,PPSR secured party security interest,Industry breakdown of new PPSR registrations,,CSV,2023-08-11T11:34:08.617300


## Generate Search Text

In [5]:
TEXT_COLUMNS = [
    "title",
    "description",
    "publisher",
    "tags",
    "resource_names",
    "resource_descriptions",
]


datasets_df["search_text"] = (
    datasets_df[TEXT_COLUMNS]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


datasets_df[
    [
        "dataset_id",
        "title",
        "publisher",
        "formats",
        "search_text",
    ]
].head()

,dataset_id,title,publisher,formats,search_text
0,5bd7fcab-e315-42cb-8daf-50b7efc2027e,ABN Bulk Extract,Australian Business Register,"CSV, XML, ZIP",ABN Bulk Extract ABN Lookup Bulk Data - Austra...
1,3a3661a8-ee5d-4165-be5c-018634ca7202,ABN Lookup web services,Australian Business Register,XML,ABN Lookup web services The ABN Lookup web ser...
2,1ca29b03-220d-4c19-b34e-c7932cd4508d,Victorian Government Schools ABNs,Department of Education,XLSX,Victorian Government Schools ABNs The Australi...
3,7c356598-b048-4c1e-862f-c936041e3d81,Research and Development tax incentive,Australian Taxation Office,XLSX,Research and Development tax incentive Data ab...
4,42f46baa-b642-48b2-9f26-fc24d09d9320,Industry breakdown of PPSR registrations,Australian Financial Security Authority,CSV,Industry breakdown of PPSR registrations The P...


## Define Keyword Ranking

In [6]:
KEYWORDS = {
    "abn": 5,
    "acn": 5,
    "australian business number": 5,
    "australian company number": 5,

    "business name": 4,
    "company name": 4,
    "legal name": 4,
    "trading name": 4,

    "company": 3,
    "companies": 3,
    "licence holder": 3,
    "license holder": 3,
    "supplier": 3,
    "contractor": 3,

    "licence": 2,
    "license": 2,
    "licensee": 2,
    "register": 2,
    "registered": 2,
    "address": 2,
    "charity": 2,
    "auditor": 2,

    "postcode": 1,
}


NEGATIVE_KEYWORDS = {
    "statistics": -3,
    "aggregate": -3,
    "aggregated": -3,
    "business counts": -4,
    "survey": -2,
}

## Calculate Keyword Score

In [7]:
def keyword_score(text):
    text = str(text).lower()

    score = 0
    matched = []

    for term, weight in KEYWORDS.items():
        if term in text:
            score += weight
            matched.append(term)

    for term, weight in NEGATIVE_KEYWORDS.items():
        if term in text:
            score += weight

    return max(score, 0), matched


keyword_results = datasets_df["search_text"].apply(keyword_score)

datasets_df["keyword_score"] = keyword_results.apply(lambda x: x[0])
datasets_df["matched_keywords"] = keyword_results.apply(lambda x: x[1])


datasets_df[
    [
        "title",
        "publisher",
        "formats",
        "keyword_score",
        "matched_keywords",
    ]
].sort_values(
    "keyword_score",
    ascending=False,
).head(30)

,title,publisher,formats,keyword_score,matched_keywords
10,ASIC - Company Dataset,Australian Securities and Investments Commissi...,"CSV, ZIP",32,"[abn, acn, australian business number, austral..."
0,ABN Bulk Extract,Australian Business Register,"CSV, XML, ZIP",30,"[abn, acn, australian business number, busines..."
9,ASIC – Australian Financial Services Authorise...,Australian Securities and Investments Commissi...,"CSV, XLSX",25,"[abn, acn, business name, licence, license, li..."
8,ASIC – Credit Licensee Dataset,Australian Securities and Investments Commissi...,"CSV, TSV, XLSX",23,"[abn, acn, business name, licence, license, li..."
7,ASIC - Australian Financial Services Licensee ...,Australian Securities and Investments Commissi...,"CSV, TSV, XLSX",21,"[abn, acn, licence, license, licensee, registe..."
6,ASIC - Business Names Dataset,Australian Securities and Investments Commissi...,"CSV, ZIP",19,"[abn, australian business number, business nam..."
456,[Superceded] Intellectual Property Government ...,IP Australia,"CSV, XLSX",17,"[abn, acn, company, license, licensee]"
5,ASIC – Credit Representative Dataset,Australian Securities and Investments Commissi...,"CSV, TSV, XLSX",17,"[abn, acn, license, licensee, register, postcode]"
11,ASIC – Financial Advisers Dataset,Australian Securities and Investments Commissi...,"CSV, XLSX",16,"[abn, licence, license, licensee, register, ad..."
447,Victorian liquor licences split by licence type,Liquor Control Victoria,CSV,15,"[business name, licence, license, licensee, re..."


In [8]:
## Load Embedding Model

In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

/Users/ario/Documents/firmable/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10622.93it/s]


## Define Semantic Target

In [10]:
TARGET = """
A dataset containing identifiable Australian businesses or companies.

Useful information includes ABN, ACN, legal name, trading name,
business name, entity type, registration status, licence information,
website, industry code, registration date, business address,
locality, state and postcode.

The dataset may contain registered businesses, licence holders,
suppliers, contractors, employers, builders, charities,
auditors or other identifiable business entities.
"""

## Calculate Embedding Score

In [11]:
dataset_embeddings = model.encode(
    datasets_df["search_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
)

target_embedding = model.encode(
    TARGET,
    normalize_embeddings=True,
)

datasets_df["embedding_score"] = (
    dataset_embeddings @ target_embedding
)


datasets_df[
    [
        "title",
        "keyword_score",
        "embedding_score",
    ]
].sort_values(
    "embedding_score",
    ascending=False,
).head(30)

Batches: 100%|██████████| 15/15 [00:01<00:00, 11.32it/s]


,title,keyword_score,embedding_score
6,ASIC - Business Names Dataset,19,0.679771
4,Industry breakdown of PPSR registrations,14,0.678452
0,ABN Bulk Extract,30,0.669006
2,Victorian Government Schools ABNs,12,0.654716
14,Australian Business Register - Statistics 2014...,9,0.633207
1,ABN Lookup web services,12,0.631144
26,Business establishments and jobs data by busin...,0,0.630809
30,Business establishments and jobs data by busin...,0,0.623617
21,Business establishments and jobs data by busin...,0,0.620403
51,Business establishments location and industry ...,2,0.620399


## Calculate Hybrid Score

In [12]:
datasets_df["keyword_score_normalized"] = (
    datasets_df["keyword_score"]
    / datasets_df["keyword_score"].max()
)

datasets_df["confidence"] = (
    0.60 * datasets_df["keyword_score_normalized"]
    + 0.40 * datasets_df["embedding_score"]
)

datasets_df = datasets_df.sort_values(
    "confidence",
    ascending=False,
).reset_index(drop=True)


datasets_df["reason"] = datasets_df.apply(
    lambda row:
        f"Matched keywords: {', '.join(row['matched_keywords']) or 'none'}; "
        f"semantic similarity: {row['embedding_score']:.3f}",
    axis=1,
)


datasets_df[
    [
        "title",
        "keyword_score",
        "keyword_score_normalized",
        "embedding_score",
        "confidence",
        "reason"
    ]
].head(50)

,title,keyword_score,keyword_score_normalized,embedding_score,confidence,reason
0,ASIC - Company Dataset,32,1.00000,0.591032,0.836413,"Matched keywords: abn, acn, australian busines..."
1,ABN Bulk Extract,30,0.93750,0.669006,0.830103,"Matched keywords: abn, acn, australian busines..."
2,ASIC – Australian Financial Services Authorise...,25,0.78125,0.533293,0.682067,"Matched keywords: abn, acn, business name, lic..."
3,ASIC – Credit Licensee Dataset,23,0.71875,0.565838,0.657585,"Matched keywords: abn, acn, business name, lic..."
4,ASIC - Business Names Dataset,19,0.59375,0.679771,0.628158,"Matched keywords: abn, australian business num..."
5,ASIC - Australian Financial Services Licensee ...,21,0.65625,0.515370,0.599898,"Matched keywords: abn, acn, licence, license, ..."
6,ASIC – Credit Representative Dataset,17,0.53125,0.578244,0.550048,"Matched keywords: abn, acn, license, licensee,..."
7,Industry breakdown of PPSR registrations,14,0.43750,0.678452,0.533881,"Matched keywords: acn, australian company numb..."
8,ASIC - Registered Auditor Dataset,14,0.43750,0.617407,0.509463,"Matched keywords: acn, register, registered, a..."
9,[Superceded] Intellectual Property Government ...,17,0.53125,0.472491,0.507746,"Matched keywords: abn, acn, company, license, ..."


## Top 50 Ranked Datasets

In [13]:
top_50 = datasets_df.head(50)[
    [
        "dataset_id",
        "title",
        "publisher",
        "formats",
        "confidence",
        "reason",
    ]
].copy()

top_50["confidence"] = top_50["confidence"].round(4)

pd.set_option("display.max_colwidth", 120)

top_50

,dataset_id,title,publisher,formats,confidence,reason
0,7b8656f9-606d-4337-af29-66b89b2eeefb,ASIC - Company Dataset,Australian Securities and Investments Commission (ASIC),"CSV, ZIP",0.8364,"Matched keywords: abn, acn, australian business number, australian company number, company name, company, companies,..."
1,5bd7fcab-e315-42cb-8daf-50b7efc2027e,ABN Bulk Extract,Australian Business Register,"CSV, XML, ZIP",0.8301,"Matched keywords: abn, acn, australian business number, business name, legal name, trading name, register, postcode;..."
2,a7bbbf64-e2ef-4d96-9fb4-0aea73ada2af,ASIC – Australian Financial Services Authorised Representative Dataset,Australian Securities and Investments Commission (ASIC),"CSV, XLSX",0.6821,"Matched keywords: abn, acn, business name, licence, license, licensee, register, address, postcode; semantic similar..."
3,fa0b0d71-b8b8-4af8-bc59-0b000ce0d5e4,ASIC – Credit Licensee Dataset,Australian Securities and Investments Commission (ASIC),"CSV, TSV, XLSX",0.6576,"Matched keywords: abn, acn, business name, licence, license, licensee, register, postcode; semantic similarity: 0.566"
4,bc515135-4bb6-4d50-957a-3713709a76d3,ASIC - Business Names Dataset,Australian Securities and Investments Commission (ASIC),"CSV, ZIP",0.6282,"Matched keywords: abn, australian business number, business name, companies, register; semantic similarity: 0.680"
5,ab7eddce-84df-4098-bc8f-500d0d9776d1,ASIC - Australian Financial Services Licensee Dataset,Australian Securities and Investments Commission (ASIC),"CSV, TSV, XLSX",0.5999,"Matched keywords: abn, acn, licence, license, licensee, register, address, postcode; semantic similarity: 0.515"
6,a3b6d445-3a00-409a-8b49-53dc62b823bf,ASIC – Credit Representative Dataset,Australian Securities and Investments Commission (ASIC),"CSV, TSV, XLSX",0.5500,"Matched keywords: abn, acn, license, licensee, register, postcode; semantic similarity: 0.578"
7,42f46baa-b642-48b2-9f26-fc24d09d9320,Industry breakdown of PPSR registrations,Australian Financial Security Authority,CSV,0.5339,"Matched keywords: acn, australian company number, company, register, registered; semantic similarity: 0.678"
8,4117fdec-3eff-44a1-9b78-869f7ec5c409,ASIC - Registered Auditor Dataset,Australian Securities and Investments Commission (ASIC),"CSV, TSV, XLSX",0.5095,"Matched keywords: acn, register, registered, address, auditor, postcode; semantic similarity: 0.617"
9,6a40d5a7-5886-4a96-8bca-9260ec4479ed,[Superceded] Intellectual Property Government Open Data 2016,IP Australia,"CSV, XLSX",0.5077,"Matched keywords: abn, acn, company, license, licensee; semantic similarity: 0.472"


In [14]:
manual_review = (
    top_50
    .sample(
        n=20,
        random_state=20,
    )
    .copy()
    # .reset_index(drop=True)
)

manual_review["is_relevant"] = pd.NA
manual_review["review_notes"] = ""

manual_review

,dataset_id,title,publisher,formats,confidence,reason,is_relevant,review_notes
12,3a3661a8-ee5d-4165-be5c-018634ca7202,ABN Lookup web services,Australian Business Register,XML,0.4775,"Matched keywords: abn, acn, register; semantic similarity: 0.631",<NA>,
14,37390371-9da4-42d8-9304-649c50090611,ASIC – Self-Managed Super Fund Auditor Dataset,Australian Securities and Investments Commission (ASIC),"CSV, XLSX",0.4338,"Matched keywords: abn, licence, register, auditor, postcode; semantic similarity: 0.522",<NA>,
4,bc515135-4bb6-4d50-957a-3713709a76d3,ASIC - Business Names Dataset,Australian Securities and Investments Commission (ASIC),"CSV, ZIP",0.6282,"Matched keywords: abn, australian business number, business name, companies, register; semantic similarity: 0.680",<NA>,
8,4117fdec-3eff-44a1-9b78-869f7ec5c409,ASIC - Registered Auditor Dataset,Australian Securities and Investments Commission (ASIC),"CSV, TSV, XLSX",0.5095,"Matched keywords: acn, register, registered, address, auditor, postcode; semantic similarity: 0.617",<NA>,
45,e761c394-a103-4243-aad2-d47f2d8fcf8d,Business establishments per CLUE industry in municipality,City of Melbourne Open Data,"CSV, JSON",0.2454,Matched keywords: none; semantic similarity: 0.614,<NA>,
1,5bd7fcab-e315-42cb-8daf-50b7efc2027e,ABN Bulk Extract,Australian Business Register,"CSV, XML, ZIP",0.8301,"Matched keywords: abn, acn, australian business number, business name, legal name, trading name, register, postcode;...",<NA>,
25,c2524c87-cea4-4636-acac-599a82048a26,Corporate Tax Transparency,Australian Taxation Office,XLSX,0.3848,"Matched keywords: abn, company, companies; semantic similarity: 0.446",<NA>,
24,6f1cc305-a8b2-4748-bd40-c00754ad6504,Victorian liquor licences by location,Liquor Control Victoria,"CSV, XLSX",0.3892,"Matched keywords: business name, licence, license, address, postcode; semantic similarity: 0.457",<NA>,
47,8a1ad782-3d60-4c34-8cd8-f11c031947b7,Fair Jobs Code Registers,"Department of Jobs, Skills, Industry and Regions",CSV,0.2444,"Matched keywords: supplier, contractor, register; semantic similarity: 0.236",<NA>,
23,b69d57f5-f9e0-4935-b6f4-1008f979dc94,[Superseded] Intellectual Property Government Open Data 2015,IP Australia,"CSV, XLSX, ZIP",0.3922,"Matched keywords: abn, acn, license, licensee; semantic similarity: 0.324",<NA>,


### Review Result

All sampled datasets are business-related, but the following datasets do not provide identifiable entities. They contain aggregated data rather than entity-level records:

- 45 — Business establishments per CLUE industry in municipality
- 36 — Annual Report CBS
- 42 — Business establishments and jobs data by business size and CLUE industry in municipality

**False positives:** 3 out of 20 (**15%**).

**Observation:** Most datasets ranked above 30 are highly relevant, while the positive rate decreases after rank 30. This indicates that the ranking approach successfully prioritised the most relevant entity-level datasets near the top.

In [15]:
top_50

,dataset_id,title,publisher,formats,confidence,reason
0,7b8656f9-606d-4337-af29-66b89b2eeefb,ASIC - Company Dataset,Australian Securities and Investments Commission (ASIC),"CSV, ZIP",0.8364,"Matched keywords: abn, acn, australian business number, australian company number, company name, company, companies,..."
1,5bd7fcab-e315-42cb-8daf-50b7efc2027e,ABN Bulk Extract,Australian Business Register,"CSV, XML, ZIP",0.8301,"Matched keywords: abn, acn, australian business number, business name, legal name, trading name, register, postcode;..."
2,a7bbbf64-e2ef-4d96-9fb4-0aea73ada2af,ASIC – Australian Financial Services Authorised Representative Dataset,Australian Securities and Investments Commission (ASIC),"CSV, XLSX",0.6821,"Matched keywords: abn, acn, business name, licence, license, licensee, register, address, postcode; semantic similar..."
3,fa0b0d71-b8b8-4af8-bc59-0b000ce0d5e4,ASIC – Credit Licensee Dataset,Australian Securities and Investments Commission (ASIC),"CSV, TSV, XLSX",0.6576,"Matched keywords: abn, acn, business name, licence, license, licensee, register, postcode; semantic similarity: 0.566"
4,bc515135-4bb6-4d50-957a-3713709a76d3,ASIC - Business Names Dataset,Australian Securities and Investments Commission (ASIC),"CSV, ZIP",0.6282,"Matched keywords: abn, australian business number, business name, companies, register; semantic similarity: 0.680"
5,ab7eddce-84df-4098-bc8f-500d0d9776d1,ASIC - Australian Financial Services Licensee Dataset,Australian Securities and Investments Commission (ASIC),"CSV, TSV, XLSX",0.5999,"Matched keywords: abn, acn, licence, license, licensee, register, address, postcode; semantic similarity: 0.515"
6,a3b6d445-3a00-409a-8b49-53dc62b823bf,ASIC – Credit Representative Dataset,Australian Securities and Investments Commission (ASIC),"CSV, TSV, XLSX",0.5500,"Matched keywords: abn, acn, license, licensee, register, postcode; semantic similarity: 0.578"
7,42f46baa-b642-48b2-9f26-fc24d09d9320,Industry breakdown of PPSR registrations,Australian Financial Security Authority,CSV,0.5339,"Matched keywords: acn, australian company number, company, register, registered; semantic similarity: 0.678"
8,4117fdec-3eff-44a1-9b78-869f7ec5c409,ASIC - Registered Auditor Dataset,Australian Securities and Investments Commission (ASIC),"CSV, TSV, XLSX",0.5095,"Matched keywords: acn, register, registered, address, auditor, postcode; semantic similarity: 0.617"
9,6a40d5a7-5886-4a96-8bca-9260ec4479ed,[Superceded] Intellectual Property Government Open Data 2016,IP Australia,"CSV, XLSX",0.5077,"Matched keywords: abn, acn, company, license, licensee; semantic similarity: 0.472"


In [16]:
deliverable = top_50.rename(columns={
    "publisher": "publishing_organisation",
    "formats": "resource_formats",
})

assert len(deliverable) == 50

output_path = OUTPUT_DIR / "ranked_shortlist_50.csv"

deliverable.to_csv(
    output_path,
    index=False,
)

print("Saved:", output_path)

Saved: /Users/ario/Documents/firmable/outputs/part1/ranked_shortlist_50.csv
